In [1]:
import mne
import numpy as np
import plotly.express as px
import plotly.io as pio
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols, wls
pio.renderers.default = 'notebook'
from crosci.biomarkers import (
    DFA,
    fEI
)
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'code'))
from compute_criticality import load_all, criticality_dir_for

In [ ]:
df = load_all(criticality_dir_for('_matchsample_v1'))
df = df[~df['bad']]
adf = df.groupby(['task', 'band', 'channel'])[['dfa', 'fei']].aggregate('mean').reset_index()

In [3]:
dataset = "ds003969"
bids_root = Path('/home/jackhoyle') / 'mne_data' / dataset
subject_info = pd.read_csv(bids_root / 'participants.tsv', sep='\t')
subject_info['subject'] = subject_info['participant_id'].str.removeprefix('sub-')
df = df.merge(subject_info[['subject', 'group']], on='subject', how='left', validate='many_to_one')

## Session order

Each block is a separate BDF file, and its header records when the recording started (`meas_date`). Ranking the four blocks of a subject by start time gives `time` = 0 (first block) to 3 (last).
Every subject follows one of two orders, all recorded on a single day: `med1breath > med2 > think1 > think2` or `think1 > think2 > med1breath > med2`.
This matches the `first_session` column of participants.tsv for all 98 subjects (where `thought` means thinking).
Each block lasts about 10 min, and the two blocks of a pair run back to back (median under 1 min between them).
Between the pairs there is a median of 66 min (range 47–89) from the end of the second block to the start of the third, similar across orders and groups.
The README describes alternating blocks, but the timestamps show this paired structure.
`time` is ranked over all four recorded blocks, so it is still correct for a subject whose block is missing from the criticality results.

The regressions control for order with two dummies instead of a linear `time`:
- `second_pair`: 1 for the pair recorded after the ~75 min break. It varies within each task (by `order`), so it is identifiable alongside task.
- `second_in_pair`: 1 for med2 and think2. These always come second in their pair, so this is not separable from task itself.
  Task is therefore coded as `med_task * second_in_pair`, which spans the same four cells as `C(task)`:
  `med_task:second_in_pair` = (med2 − med1breath) − (think2 − think1), i.e. the rise from first to second block in meditation beyond the same rise in thinking.
  A "deepening of meditation" effect predicts this is positive (or negative, for H2's predicted decrease); a generic time-on-task effect predicts it is zero.

In [4]:
import mne

order_rows = []
for subject in sorted(df['subject'].unique()):
    for bdf_path in sorted((bids_root / f'sub-{subject}' / 'eeg').glob(f'sub-{subject}_task-*_eeg.bdf')):
        info = mne.io.read_raw_bdf(bdf_path, preload=False, verbose='error').info
        task = bdf_path.name.split('_task-')[1].split('_')[0]
        order_rows.append(dict(subject=subject, task=task, start=info['meas_date']))
session_order = pd.DataFrame(order_rows)
session_order['time'] = session_order.groupby('subject')['start'].rank(method='first').astype(int) - 1
session_order['order'] = session_order['subject'].map(
    session_order.sort_values('start').groupby('subject')['task'].first().map(
        {'med1breath': 'med_first', 'think1': 'think_first'}))

# every subject has 4 blocks on one day, in one of the two orders
assert (session_order.groupby('subject')['time'].apply(sorted).map(tuple) == (0, 1, 2, 3)).all()
assert session_order['order'].notna().all()
assert (session_order.groupby('subject')['start'].apply(lambda s: s.dt.date.nunique()) == 1).all()

session_order['second_pair'] = (session_order['time'] >= 2).astype(int)
session_order['second_in_pair'] = (session_order['time'] % 2).astype(int)
# med2 and think2 are always the second block of their pair
assert (session_order['second_in_pair'] == session_order['task'].isin(['med2', 'think2'])).all()

df = df.merge(session_order[['subject', 'task', 'time', 'order', 'second_pair', 'second_in_pair']],
              on=['subject', 'task'], how='left', validate='many_to_one')
assert df['time'].notna().all()
df['med_task'] = df['task'].isin(['med1breath', 'med2']).astype(int)
df['meditator'] = (df['group'] != 'ctr').astype(int)

print(session_order.sort_values(['subject', 'time']).groupby('subject')['task'].agg(' > '.join).value_counts())
pd.crosstab(df.drop_duplicates('subject')['group'], df.drop_duplicates('subject')['order'], margins=True)

task
med1breath > med2 > think1 > think2    28
think1 > think2 > med1breath > med2    27
Name: count, dtype: int64


order,med_first,think_first,All
group,,,
ctr,8,7,15
htr,7,6,13
sny,7,7,14
vip,6,7,13
All,28,27,55


# results

In [6]:
# \begin{itemize}
#   \item \textbf{$H_1$}: There is no trait difference in criticality measures (DFA and fEI) between groups.
#   \item \textbf{$H_2$}: Meditators show an increase in DFA criticality over time whilst meditating, i.e. 
#       from \texttt{med1think} to \texttt{med2}.
# \end{itemize}

In [7]:
bands = ['alpha', 'beta', 'gamma']

In [8]:
# H1
# in line with previous analysis we define the treat as the med2, think2 average, in line with original paper

for band in bands:
    bdf = df[(df['band'] == band) & (~df['group'].isin(['tm'])) & (~df['task'].isin(
        ['med1breath', 'think1']))].groupby(
        ['task', 'subject', 'group', 'meditator', 'med_task', 'second_pair', 'second_in_pair']
        )[['dfa', 'fei']].aggregate('mean').reset_index()

    lm = ols(
        'dfa ~  C(group) + second_pair', 
        data=bdf).fit(cov_type='cluster', cov_kwds={'groups': pd.factorize(bdf['subject'])[0]})
    
    print(f"Band: {band}")
    print(lm.summary())

Band: alpha
                            OLS Regression Results                            
Dep. Variable:                    dfa   R-squared:                       0.080
Model:                            OLS   Adj. R-squared:                  0.025
Method:                 Least Squares   F-statistic:                     1.474
Date:                Tue, 20 Oct 2026   Prob (F-statistic):              0.225
Time:                        10:31:55   Log-Likelihood:                 53.311
No. Observations:                  72   AIC:                            -96.62
Df Residuals:                      67   BIC:                            -85.24
Df Model:                           4                                         
Covariance Type:              cluster                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept           0.7957    

In [15]:
# H2

for band in bands:
    bdf = df[(df['band'] == band) & (~df['group'].isin(['tm', 'ctr']))].groupby(
        ['task', 'subject', 'group', 'meditator', 'med_task', 'second_pair', 'second_in_pair']
        )[['dfa', 'fei']].aggregate('mean').reset_index()

    lm = ols(
        'dfa ~ C(med_task) * C(second_in_pair) + second_pair', 
        data=bdf).fit(cov_type='cluster', cov_kwds={'groups': pd.factorize(bdf['subject'])[0]})
    
    print(f"Band: {band}")
    print(lm.summary())# Use one regression that we think best explains the data, and test hypotheses based on this


Band: alpha
                            OLS Regression Results                            
Dep. Variable:                    dfa   R-squared:                       0.037
Model:                            OLS   Adj. R-squared:                  0.003
Method:                 Least Squares   F-statistic:                     1.445
Date:                Tue, 20 Oct 2026   Prob (F-statistic):              0.237
Time:                        11:45:49   Log-Likelihood:                 76.029
No. Observations:                 117   AIC:                            -142.1
Df Residuals:                     112   BIC:                            -128.2
Df Model:                           4                                         
Covariance Type:              cluster                                         
                                              coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------

Deepening vs time-on-task, meditators (htr, sny, vip pooled) vs controls.
`med_task:second_in_pair` is the extra second-block rise in meditation over thinking for controls; 
`meditator:med_task:second_in_pair` is how much larger that is in meditators. 
Note that for meditators med2 is their tradition-specific practice, while for controls it is breath counting again.